In [ ]:
!python 01_lithography/photolithography_baseline.py
#测试代码，如果没有报错，说明环境配置成功了。

## Stage 1: Lithography (光刻阶段)

**输入**: 掩膜 (mask.npy)  
**输出**: resist.npy - 显影后的光刻胶图像

### 工艺参数说明：
- **wavelength_nm**: 光源波长 (193nm ArF激光)
- **NA**: 数值孔径，影响分辨率
- **sigma_in/sigma_out**: 环形光源参数 (部分相干照明)
- **peb_blur_nm**: 曝光后烘烤 (PEB) 高斯模糊半径
- **develop_threshold**: 显影阈值 (0-1)
- **dose/defocus**: 工艺窗口参数

### 输出内容：
- `01_lithography/outputs/stage1_poly_litho/resist.npy` - 最终光刻胶
- 工艺窗口分析 (CD vs Dose/Defocus)
- 过度蚀刻指标

In [ ]:
!python 01_lithography/run_project.py --config configs/lithography_config.json
#光刻部分运行，生成光刻图像和相关数据。输出文件夹为outputs/stage1_poly_litho。可以在该文件夹中查看生成的图像和数据。
#也可以自行更改输出文件夹，方法为在后面添加--output_dir your_output_folder_name

## Stage 2: Etch (刻蚀阶段)

**输入**: resist.npy (从 Stage1 光刻得到)  
**输出**: 
- etched_openings.npy (蚀刻开口掩膜)
- summary.json (蚀刻摘要)

### 工艺参数说明：
- **pixel_um**: 像素尺寸
- **base_rate_Apm**: 基础刻蚀速率 (Å/min)
- **film_thickness_A**: 膜厚 (Å)
- **pressure_mTorr**: 等离子体压力
- **rf_power_W**: RF 功率
- **magnetic_field_mT**: 磁场强度

### 过程建模：
- **各向异性/各向同性权衡** - 压力、磁场控制刻蚀轮廓
- **宏观加载效应** - 开口面积大小影响总体刻蚀速率
- **微观加载效应** - 小特征处刻蚀速率降低

In [ ]:
# ============================================================
# Stage 2: Etch (刻蚀)
# ============================================================
import os
import shutil
import subprocess

outdir = "outputs/stage2_etch"

# Clean existing output contents to avoid collisions with stale artifacts.
if os.path.isdir(outdir):
    for name in os.listdir(outdir):
        path = os.path.join(outdir, name)
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)
else:
    os.makedirs(outdir, exist_ok=True)

# Accepted params from A/B workflow tuning:
cmd = [
    "python", "02_process_stages/stage2_etch.py",
    "--resist", "outputs/stage1_poly_litho/resist.npy",
    "--outdir", outdir,
    "--pixel_um", "0.02",
    "--base_rate_Apm", "5500.0",
    "--film_thickness_A", "3000.0",
    "--pressure_mTorr", "12.0",
    "--rf_power_W", "500.0",
    "--magnetic_field_mT", "50.0"
]

print("[CMD]", " ".join(cmd))
try:
    subprocess.check_call(cmd)
    print("✅ Stage2 Etch 完成")
except Exception as e:
    print(f"⚠️ 错误: {e}")

## Stage 3: CVD (化学气相沉积阶段)

**输入**: etched_openings.npy (从 Stage2 蚀刻得到)  
**输出**: 
- depth_map_um.npy (沉积厚度分布图)
- summary.json (步阶覆盖指标)

### 工艺参数说明：
- **沉积方向**: 各向同性 / 各向异性
- **步阶覆盖**: 凹陷处覆盖率
- **间隙填充**: 高宽比限制
- **沉积速率**: 依赖于角度和位置

### 过程建模：
- **角度相关沉积** - 陡峭表面和平面沉积速率不同
- **间隙填充能力** - 实际应用选择：ALD vs PECVD vs LPCVD
- **步阶覆盖效率** - 评估沉积均匀性

In [ ]:

# ============================================================
# Stage 3: CVD (化学气相沉积)
# ============================================================
import os
import subprocess

os.makedirs("outputs/stage3_cvd", exist_ok=True)

cmd = [
    "python", "02_process_stages/stage3_cvd.py",
    "--openings", "outputs/stage2_etch/etched_openings.npy",
    "--outdir", "outputs/stage3_cvd",
    "--pixel_um", "0.02",
    "--trench_depth_um", "0.3",
    "--target_thickness_um", "0.55",
    "--conformality", "0.95",
    "--regime", "mass_transport",
    "--temperature_C", "400.0",
    "--hdp_cycles", "0",
    "--hdp_sputter_frac", "0.25"
]

print("[CMD]", " ".join(cmd))
try:
    subprocess.check_call(cmd)
    print("✅ Stage3 CVD 完成")
except Exception as e:
    print(f"⚠️ 错误: {e}")


## Stage 4: Ion Implantation + Thermal Anneal (离子注入+热退火阶段)

**输入**: ACTIVE/NIMP/PIMP 掩膜 (从 masks/ 目录)  
**输出**: 
- 掺杂浓度分布图
- metrics_stage4.json (植入指标)

### 工艺参数说明：
- **dopant**: 掺杂剂类型 (B/P/As)
- **dose_cm2**: 植入剂量 (1e13 - 1e16 cm⁻²)
- **energy_keV**: 植入能量 (1-1000 keV)
- **anneal_method**: 退火方式 (RTA/furnace)
- **anneal_T_C**: 退火温度 (°C)
- **anneal_time_s**: 退火时间 (秒)
- **channeling**: 是否考虑沟道化效应

### 过程建模：
- **离子射程** - Rp 和 ΔRp（高斯分布）
- **沟道尾效应** - 沿晶向加速离子穿透
- **热扩散** - D ∝ exp(-Ea/kT)，影响结深

In [ ]:
# ============================================================
# Stage 4: Ion Implantation + Thermal Anneal (植入+退火)
# ============================================================
import json, os, shutil, subprocess

outdir = "outputs/stage4_implant_thermal"
if os.path.isdir(outdir):
    for name in os.listdir(outdir):
        path = os.path.join(outdir, name)
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)
else:
    os.makedirs(outdir, exist_ok=True)

cmd = [
    "python", "02_process_stages/stage4_thermal_implant.py",
    "--outdir", outdir,
    "--dopant", "B",
    "--dose_cm2", "2e15",
    "--energy_keV", "4",
    "--anneal_method", "rta",
    "--anneal_T_C", "1050",
    "--anneal_time_s", "10",
    "--tilt_deg", "7",
    "--screen_oxide_nm", "10",
    "--channeling",
    "--pre_amorphous",
]
subprocess.check_call(cmd)
with open(os.path.join(outdir, "metrics.json"), "r", encoding="utf-8") as f:
    metrics = json.load(f)
print(
    "Stage4 done: xj=%.4f um, Rs=%.2f ohm/sq, act=%.4f" % (
        metrics["junction_depth_um"],
        metrics["sheet_resistance_ohm_per_sq"],
        metrics["activation_fraction"],
    )
)

## Stage 5: Metallization + CMP (金属化+化学机械抛光阶段)

**输入**: 
- depth_map_um.npy (从 Stage3 CVD 得到)
- etched_openings.npy (从 Stage2 蚀刻得到)

**输出**: 
- 平坦化后的表面拓扑图
- metrics_stage5.json (CMP 指标)

### 工艺参数说明：
- **overburden_nm**: 金属沉积厚度 (nm)
- **target_overburden_nm**: CMP 目标厚度 (nm)
- **pattern_density_window**: 图案密度范围

### 过程建模：
- **金属沉积** - Cu 大马士革工艺
- **CMP 去除速率** - Preston 方程近似 (Rate ∝ Pressure × Velocity)
- **图案密度效应** - 稀疏区抛光速度不同
- **选择性效应** - 金属 vs. 绝缘体去除率比
- **盘蚀/侵蚀** - 重要的缺陷模式

In [ ]:
# ============================================================
# Stage 5: Metallization + CMP (金属化 + 化学机械抛光)
# ============================================================
import json, os, shutil, subprocess

outdir = "outputs/stage5_metallization_cmp"
if os.path.isdir(outdir):
    for name in os.listdir(outdir):
        path = os.path.join(outdir, name)
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)
else:
    os.makedirs(outdir, exist_ok=True)

cmd = [
    "python", "02_process_stages/stage5_metallization_cmp.py",
    "--depth_map", "outputs/stage3_cvd/depth_map_um.npy",
    "--features", "outputs/stage2_etch/etched_openings.npy",
    "--outdir", outdir,
    "--barrier_nm", "10",
    "--seed_nm", "50",
    "--overburden_um", "0.8",
    "--cmp_target_overburden_um", "0.12",
    "--dishing_strength_um", "0.05",
    "--erosion_strength_um", "0.02",
]
subprocess.check_call(cmd)
with open(os.path.join(outdir, "metrics.json"), "r", encoding="utf-8") as f:
    metrics = json.load(f)
print(
    "Stage5 done: topo_range=%.6f um, copper_mean=%.6f um" % (
        metrics["topo_range_um"],
        metrics["copper_mean_um"],
    )
)

## Stage 6: Device Proxy (器件特性提取阶段)

**输入**: 
- etch_summary.json (从 Stage2 得到)
- implant_metrics.json (从 Stage4 得到)
- cmp_metrics.json (从 Stage5 得到)

**输出**: 
- Vt (阈值电压) 预测
- Ileak (漏电流) 估算
- Rsheet (片电阻) / Rvia (通孔电阻)
- metrics_stage6.json (器件性能指标汇总)

### 工艺参数说明：
- **tox_nm**: 栅氧厚度 (nm)
- **W_nm**: 器件宽度 (nm)

### 过程建模：
- **紧凑模型** - 参数化查表或解析模型
- **W/L 效应** - 短 Channel 效应对 Vt 的影响
- **结深影响** - Xj 决定片电阻 Rsheet
- **金属寄生电阻** - 线宽和导电性影响通孔电阻

In [ ]:
# ============================================================
# Stage 6: Device Proxy (器件特性提取)
# ============================================================
import json, os, shutil, subprocess

outdir = "outputs/stage6_device"
if os.path.isdir(outdir):
    for name in os.listdir(outdir):
        path = os.path.join(outdir, name)
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)
else:
    os.makedirs(outdir, exist_ok=True)

cmd = [
    "python", "02_process_stages/stage6_device.py",
    "--etch_summary", "outputs/stage2_etch/summary.json",
    "--implant_metrics", "outputs/stage4_implant_thermal/metrics.json",
    "--cmp_metrics", "outputs/stage5_metallization_cmp/metrics.json",
    "--outdir", outdir,
    "--cd_nom_nm", "200.0",
    "--tox_nm", "2.5",
    "--temp_C", "25.0",
    "--W_um", "1.2",
]
subprocess.check_call(cmd)
with open(os.path.join(outdir, "metrics.json"), "r", encoding="utf-8") as f:
    metrics = json.load(f)
print(
    "Stage6 done: Vt=%.6f, Ioff=%.6e, Ron=%.6f, Rint=%.6f" % (
        metrics["Vt_proxy_V"],
        metrics["Ioff_proxy_A"],
        metrics["Ron_proxy_ohm"],
        metrics["interconnect_R_proxy_ohm"],
    )
)

## 完整流程汇总

✅ 所有 6 个 Stage 完成！

### 输出文件位置：
- **Stage2**: `outputs/stage2_etch/` → etched_openings.npy, summary.json
- **Stage3**: `outputs/stage3_cvd/` → depth_map_um.npy, summary.json
- **Stage4**: `outputs/stage4_implant_thermal/` → metrics_stage4.json
- **Stage5**: `outputs/stage5_metallization_cmp/` → metrics_stage5.json
- **Stage6**: `outputs/stage6_device/` → metrics_stage6.json

### 学生修改建议：
| 模块 | 改进方向 |
|------|--------|
| **stage2_etch.py** | 微加载模型和各向异性-偏差映射 |
| **stage3_cvd.py** | 实现角度相关沉积；添加 ALD vs PECVD vs LPCVD 选择 |
| **stage4_thermal_implant.py** | 替换 Rp/ΔRp 模型；实现 2D 扩散 |
| **stage5_metallization_cmp.py** | 使用 Preston 方程改进 CMP；添加选择性效应 |
| **stage6_device.py** | 用更好的紧凑模型替换代理模型 |

## 完整 7 层流程一次性执行

使用 `run_7mask_pipeline.py` 脚本，从 GDS 输入开始，依次执行：
1. GDS → 掩模转换
2. Stage1 光刻
3. Stage2 刻蚀
4. Stage4 植入+退火
5. Stage5 金属化+CMP
6. Stage6 器件提取

**配置文件**: `configs/process_flow.json`  
**输出目录**: `outputs/<timestamp>_team_run/`

In [2]:
# ============================================================
# 完整 7 层工艺流程 (End-to-End 7-Mask Pipeline)
# ============================================================
import os
import subprocess

out_root = "outputs"
before = set(os.listdir(out_root)) if os.path.isdir(out_root) else set()

print("=" * 70)
print("🚀 启动虚拟晶圆厂 7 层完整工艺流程")
print("=" * 70)
print()

cmd = [
    "python", "02_process_stages/run_7mask_pipeline.py",
    "--flow", "configs/process_flow.json",
    "--runname", "clean_final"
]

print("[CMD]", " ".join(cmd))
print()

try:
    subprocess.check_call(cmd, timeout=3600)
    print()
    print("=" * 70)
    print("✅ 完整流程成功执行！")
    print("=" * 70)
    print()

    after = set(os.listdir(out_root)) if os.path.isdir(out_root) else set()
    new_dirs = sorted([d for d in (after - before) if os.path.isdir(os.path.join(out_root, d))])
    if new_dirs:
        print("📁 新输出目录:", os.path.join(out_root, new_dirs[-1]))
    else:
        print("⚠️ 未检测到新增输出目录，请检查 runname 或权限。")

except subprocess.TimeoutExpired:
    print("⏱️ 执行超时 (>1小时)")
except Exception as e:
    print(f"❌ 执行失败: {e}")

🚀 启动虚拟晶圆厂 7 层完整工艺流程

[CMD] python 02_process_stages/run_7mask_pipeline.py --flow configs/process_flow.json --runname clean_final

[CMD] python gds_router/gds_to_masks.py --gds /workspaces/6202Project/gds/basic.gds --layer_map /workspaces/6202Project/configs/layer_map.json --outdir /workspaces/6202Project/masks --N 1024 --top_cell 
Wrote ACTIVE polygons: 40665
Wrote POLY polygons: 8040
Wrote NIMP polygons: 22820
Wrote PIMP polygons: 42865
Wrote CONT polygons: 34695
Wrote M1 polygons: 21040
Wrote V1 polygons: 1370
[CMD] python run_project.py --config config.json


/home/codespace/.local/lib/python3.12/site-packages/matplotlib/cbook.py:1719: ComplexWarning: Casting complex values to real discards the imaginary part
  return math.isfinite(val)
/home/codespace/.local/lib/python3.12/site-packages/matplotlib/cbook.py:1355: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.asarray(x, float)


[OK] Outputs saved to ./outputs/stage1_poly_litho
[Metric] Window area ≈ 125.90 (dose·nm)
[CMD] python stage2_etch.py --resist /workspaces/6202Project/outputs/20260412_141115_clean_final/stage1_poly_litho/resist.npy --outdir /workspaces/6202Project/outputs/20260412_141115_clean_final/stage2_etch --pixel_um 0.02 --base_rate_Apm 5500.0 --film_thickness_A 3000.0 --pressure_mTorr 12.0 --rf_power_W 500.0 --magnetic_field_mT 50.0
Stage 2 finished. Outputs: /workspaces/6202Project/outputs/20260412_141115_clean_final/stage2_etch
Min selectivity required (over-etch): 39.0
[CMD] python stage3_cvd.py --openings /workspaces/6202Project/outputs/20260412_141115_clean_final/stage2_etch/etched_openings.npy --outdir /workspaces/6202Project/outputs/20260412_141115_clean_final/stage3_cvd --pixel_um 0.02 --trench_depth_um 0.3 --target_thickness_um 0.55 --conformality 0.95 --regime mass_transport --temperature_C 400.0 --hdp_cycles 0 --hdp_sputter_frac 0.25
Stage 3 (CVD) finished. Outputs: /workspaces/6202P